In [25]:
# SECTION 1: REQUIRED PACKAGES AND UTILITIES


import os
import time
import random
import logging
from dataclasses import dataclass
from datetime import datetime
from collections import deque, namedtuple, Counter
from typing import List, Tuple, Dict, Optional, Union

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import chess

# Utility functions for reproducibility and learning rate scheduling
def seed_everything(seed: int = 42):
    """Set random seeds for reproducible results"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def linear_anneal(start, end, cur, total):
    """Linear annealing schedule for hyperparameters"""
    t = min(1.0, cur / float(total))
    return start + (end - start) * t

In [27]:
# SECTION 2: CHESS ENVIRONMENT IMPLEMENTATION

@dataclass
class GameState:
    """State container for chess game including timing information"""
    board: chess.Board
    white_time: float
    black_time: float
    move_count: int
    last_move_time: float
    game_start_time: float

class BulletChessEnv:
    """
    Chess environment with time controls for reinforcement learning.
    Features:
    - Bullet time controls (60 seconds default)
    - Move simulation with thinking time
    - Material evaluation and positional rewards
    - Comprehensive game state observation
    """

    def __init__(self,
                 time_limit: int = 60,
                 increment: float = 0.0,
                 simulate_think: bool = True,
                 think_lo: float = 0.80,
                 think_hi: float = 1.60):
        self.time_limit = time_limit
        self.increment = increment
        self.simulate_think = simulate_think
        self.think_lo = think_lo
        self.think_hi = think_hi

        # Standard chess piece values for reward calculation
        self.piece_values = {
            chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3,
            chess.ROOK: 5, chess.QUEEN: 9, chess.KING: 0
        }
        self.reset()

    def reset(self) -> np.ndarray:
        """Initialize new game and return initial observation"""
        self.state = GameState(
            board=chess.Board(),
            white_time=self.time_limit,
            black_time=self.time_limit,
            move_count=0,
            last_move_time=time.time(),
            game_start_time=time.time()
        )
        return self.get_observation()

    def step(self, action: Union[int, str], think_time: Optional[float] = None) -> Tuple[np.ndarray, float, bool, Dict]:
        """
        Execute one move in the environment

        Args:
            action: Either integer action index or UCI move string
            think_time: Optional override for thinking time

        Returns:
            observation, reward, done, info
        """
        if self.is_game_over():
            return self.get_observation(), 0.0, True, {"reason": "game_already_over"}

        old_board = self.state.board.copy()

        # Calculate thinking time
        if think_time is None and self.simulate_think:
            actual_think = float(np.random.uniform(self.think_lo, self.think_hi))
        else:
            actual_think = float(think_time or 0.0)

        # Deduct time from current player's clock
        if self.state.board.turn:
            self.state.white_time -= actual_think
            remaining = self.state.white_time
        else:
            self.state.black_time -= actual_think
            remaining = self.state.black_time

        # Check for time flag
        if remaining <= 0:
            reward = -20.0 if self.state.board.turn else 20.0
            return self.get_observation(), reward, True, {"reason": "time_flag"}

        # Convert action to chess move
        if isinstance(action, int):
            move = self._action_to_move(action)
        else:
            try:
                move = chess.Move.from_uci(action)
            except Exception:
                return self.get_observation(), -10.0, True, {"reason": "invalid_uci"}

        # Validate move legality
        if move not in self.state.board.legal_moves:
            return self.get_observation(), -10.0, True, {"reason": "illegal_move"}

        # Execute move
        self.state.board.push(move)
        self.state.move_count += 1
        self.state.last_move_time += actual_think

        # Add increment after move
        if self.state.board.turn:
            self.state.black_time += self.increment
        else:
            self.state.white_time += self.increment

        # Calculate reward and check for game termination
        reward, done, info = self._calc_reward(old_board, move, actual_think)
        return self.get_observation(), reward, done, info

    def is_game_over(self) -> bool:
        """Check if game has ended by any condition"""
        return (
            self.state.board.is_game_over()
            or self.state.white_time <= 0
            or self.state.black_time <= 0
        )

    def get_result(self) -> Optional[str]:
        """Get game result in standard notation (1-0, 0-1, 1/2-1/2)"""
        if not self.is_game_over():
            return None
        if self.state.white_time <= 0:
            return "0-1"
        elif self.state.black_time <= 0:
            return "1-0"
        else:
            res = self.state.board.result()
            return res if res != "*" else None

    def get_observation(self) -> np.ndarray:
        """
        Create neural network input from game state

        Returns:
            8x8x15 observation tensor:
            - Channels 0-5: White pieces (P,R,N,B,Q,K)
            - Channels 6-11: Black pieces (P,R,N,B,Q,K)
            - Channel 12: Current player to move
            - Channel 13: Current player's remaining time (normalized)
            - Channel 14: Opponent's remaining time (normalized)
        """
        b = self.state.board
        obs = np.zeros((8, 8, 15), dtype=np.float32)

        # Map chess piece types to channel indices
        piece_map = {
            chess.PAWN: 0, chess.ROOK: 1, chess.KNIGHT: 2,
            chess.BISHOP: 3, chess.QUEEN: 4, chess.KING: 5
        }

        # Encode piece positions
        for sq in chess.SQUARES:
            p = b.piece_at(sq)
            if p:
                r, c = divmod(sq, 8)
                ch = piece_map[p.piece_type] + (6 if p.color == chess.BLACK else 0)
                obs[r, c, ch] = 1.0

        # Encode game metadata
        obs[:, :, 12] = 1.0 if b.turn else 0.0  # Current player
        current_time = self.state.white_time if b.turn else self.state.black_time
        opp_time = self.state.black_time if b.turn else self.state.white_time
        obs[:, :, 13] = current_time / self.time_limit  # Normalized time
        obs[:, :, 14] = opp_time / self.time_limit

        return obs

    def get_legal_actions(self) -> List[int]:
        """Get list of legal action indices for current position"""
        return [self._move_to_action(m) for m in self.state.board.legal_moves]

    def _move_to_action(self, move: chess.Move) -> int:
        """Convert chess move to action index (from_square * 64 + to_square)"""
        return move.from_square * 64 + move.to_square

    def _action_to_move(self, action: int) -> chess.Move:
        """Convert action index back to chess move, handling pawn promotion"""
        from_sq = action // 64
        to_sq = action % 64
        move = chess.Move(from_sq, to_sq)

        # Handle pawn promotion (auto-promote to queen)
        p = self.state.board.piece_at(from_sq)
        if p and p.piece_type == chess.PAWN:
            rank = chess.square_rank(to_sq)
            if rank == 7 or rank == 0:
                move = chess.Move(from_sq, to_sq, promotion=chess.QUEEN)
        return move

    def _calc_material_delta(self, old_board: chess.Board, move: chess.Move) -> float:
        """Calculate immediate tactical reward from move"""
        reward = 0.0

        # Reward captures based on piece value
        if old_board.is_capture(move):
            captured = old_board.piece_at(move.to_square)
            if captured:
                reward += self.piece_values.get(captured.piece_type, 0) * 0.1

        # Bonus for tactical elements
        tmp = old_board.copy()
        tmp.push(move)
        if tmp.is_check():
            reward += 0.05
        if move.promotion:
            reward += 0.3
        if old_board.is_castling(move):
            reward += 0.1

        return reward

    def _calc_reward(self, old_board, move, think_time) -> Tuple[float, bool, Dict]:
        """
        Calculate reward for the current move and check game termination

        Reward structure:
        - Terminal rewards: ±15 for win/loss, 0 for draw
        - Material rewards: scaled by piece values
        - Small penalty for each move to encourage efficiency
        - Penalty for very long games
        """
        board = self.state.board

        # Handle game termination
        if board.is_game_over():
            outcome = board.outcome(claim_draw=True)
            if outcome is not None:
                if outcome.termination == chess.Termination.CHECKMATE:
                    reward = 15.0 if outcome.winner else -15.0
                    return reward, True, {"reason": "checkmate", "winner": "white" if outcome.winner else "black"}
                elif outcome.termination in {
                    chess.Termination.STALEMATE,
                    chess.Termination.INSUFFICIENT_MATERIAL,
                    chess.Termination.SEVENTYFIVE_MOVES,
                    chess.Termination.FIVEFOLD_REPETITION,
                    chess.Termination.FIFTY_MOVES,
                    chess.Termination.THREEFOLD_REPETITION
                }:
                    return 0.0, True, {"reason": "draw", "type": outcome.termination.name}
                else:
                    reward = 15.0 if outcome.winner else -15.0
                    return reward, True, {"reason": "timeout_or_resignation",
                                          "winner": "white" if outcome.winner else "black"}
            return 0.0, True, {"reason": "unknown_game_over"}

        # Calculate step reward
        r = self._calc_material_delta(old_board, move)
        r = np.clip(r, -1.5, 1.5)
        r -= 0.02  # Small move cost to encourage efficiency

        # Penalty for very long games
        if self.state.move_count > 120:
            r -= 0.01

        r = float(np.clip(r, -2.0, 2.0))
        return r, False, {"reason": "continue", "material": r}

In [28]:
# SECTION 3: DEEP Q-NETWORK IMPLEMENTATION

class DuelingQNet(nn.Module):
    """
    Dueling DQN architecture for chess position evaluation

    Architecture:
    - Convolutional layers for spatial pattern recognition
    - Separate value and advantage streams
    - Batch normalization and dropout for regularization
    """

    def __init__(self, in_channels: int = 15, hidden: int = 512, n_actions: int = 4096, dropout: float = 0.1):
        super().__init__()
        self.n_actions = n_actions

        # Convolutional feature extraction
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout * 0.5),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout * 0.5),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )

        conv_out = 256 * 8 * 8

        # Shared feature layer
        self.fc = nn.Sequential(
            nn.Linear(conv_out, hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )

        # Advantage stream (action preferences)
        self.adv = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden // 2, n_actions)
        )

        # Value stream (state value)
        self.val = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden // 2, 1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass implementing dueling architecture"""
        x = self.conv(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        adv = self.adv(x)
        val = self.val(x)

        # Combine value and advantage streams
        q = val + adv - adv.mean(dim=1, keepdim=True)
        return q


# Prioritized Experience Replay Components
NStepExp = namedtuple("NStepExp", ["state", "action", "reward", "next_state", "done", "mask"])

class NStepBuffer:
    """Buffer for n-step learning to improve temporal credit assignment"""

    def __init__(self, n: int, gamma: float):
        self.n = n
        self.gamma = gamma
        self.buf: List[NStepExp] = []

    def reset(self):
        self.buf.clear()

    def push(self, exp: NStepExp):
        self.buf.append(exp)

    def can_pop(self):
        return len(self.buf) >= self.n

    def pop(self) -> NStepExp:
        """Calculate n-step return and create transition"""
        R = 0.0
        next_state = None
        next_mask = None
        done = False

        # Calculate discounted n-step return
        for i, e in enumerate(self.buf[:self.n]):
            R += (self.gamma ** i) * e.reward
            if e.done:
                done = True
                next_state = e.next_state
                next_mask = e.mask
                break

        if not done:
            last = self.buf[self.n - 1]
            next_state = last.next_state
            next_mask = last.mask
            done = last.done

        first = self.buf[0]
        self.buf.pop(0)
        return NStepExp(first.state, first.action, R, next_state, done, next_mask)

    def flush_all(self):
        """Flush all remaining experiences at episode end"""
        outs = []
        while self.buf:
            outs.append(self.pop())
        return outs


class PERBuffer:
    """
    Prioritized Experience Replay buffer for more efficient learning

    Features:
    - Priority-based sampling using TD errors
    - Importance sampling weights to correct bias
    - Annealed beta parameter for bias correction
    """

    def __init__(self, capacity: int, n_actions: int, alpha: float = 0.6,
                 beta_start: float = 0.4, beta_frames: int = 1_000_000):
        self.capacity = capacity
        self.n_actions = n_actions
        self.alpha = alpha  # Priority exponent
        self.beta_start = beta_start  # Importance sampling exponent
        self.beta_frames = beta_frames

        self.pos = 0
        self.size = 0

        # Pre-allocated arrays for efficiency
        self.states = np.zeros((capacity, 8, 8, 15), dtype=np.float32)
        self.actions = np.zeros((capacity,), dtype=np.int32)
        self.rewards = np.zeros((capacity,), dtype=np.float32)
        self.next_states = np.zeros((capacity, 8, 8, 15), dtype=np.float32)
        self.dones = np.zeros((capacity,), dtype=np.bool_)
        self.next_masks = np.zeros((capacity, n_actions), dtype=np.bool_)

        self.priorities = np.zeros((capacity,), dtype=np.float32)
        self.max_priority = 1.0
        self.frame = 1

    def __len__(self):
        return self.size

    def beta(self):
        """Annealed beta for importance sampling"""
        return linear_anneal(self.beta_start, 1.0, self.frame, self.beta_frames)

    def push(self, s, a, r, ns, d, nm):
        """Add experience with maximum priority"""
        idx = self.pos
        self.states[idx] = s
        self.actions[idx] = a
        self.rewards[idx] = r
        self.next_states[idx] = ns
        self.dones[idx] = d
        self.next_masks[idx] = nm
        self.priorities[idx] = self.max_priority

        self.pos = (self.pos + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size: int):
        """Sample batch with importance weights"""
        if self.size == 0:
            raise ValueError("PERBuffer empty")

        # Calculate sampling probabilities
        prios = self.priorities[:self.size] ** self.alpha
        probs = prios / prios.sum()
        idxs = np.random.choice(self.size, batch_size, p=probs)

        # Calculate importance sampling weights
        beta = self.beta()
        self.frame += 1
        weights = (self.size * probs[idxs]) ** (-beta)
        weights = weights / weights.max()

        # Return batch as tensors
        batch = (
            torch.from_numpy(self.states[idxs]),
            torch.from_numpy(self.actions[idxs]),
            torch.from_numpy(self.rewards[idxs]),
            torch.from_numpy(self.next_states[idxs]),
            torch.from_numpy(self.dones[idxs]),
            torch.from_numpy(self.next_masks[idxs]),
            torch.from_numpy(weights.astype(np.float32)),
            torch.from_numpy(idxs.astype(np.int64)),
        )
        return batch

    def update_priorities(self, idxs: torch.Tensor, prios: torch.Tensor):
        """Update priorities based on TD errors"""
        prios = prios.detach().cpu().numpy()
        idxs = idxs.detach().cpu().numpy()
        np.maximum(prios, 1e-6, out=prios)  # Ensure minimum priority
        self.priorities[idxs] = prios
        self.max_priority = max(self.max_priority, prios.max())


class DDQNAgent:
    """
    Double Deep Q-Network agent with advanced features:
    - Dueling architecture for better value estimation
    - Prioritized experience replay for efficient learning
    - N-step learning for improved temporal credit assignment
    - Soft target updates for stability
    - Action masking for legal move enforcement
    """

    def __init__(self,
                 lr=1e-4,
                 gamma=0.99,
                 epsilon_start=1.0,
                 epsilon_end=0.05,
                 epsilon_decay=0.9996,
                 batch_size=64,
                 target_update=2000,
                 tau: float = 0.002,
                 use_soft_update: bool = True,
                 n_actions: int = 4096,
                 per_alpha: float = 0.6,
                 per_beta_start: float = 0.4,
                 per_beta_frames: int = 1_000_000,
                 n_step: int = 3,
                 dropout: float = 0.1,
                 noise_std: float = 0.01,
                 device=None):

        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Initialize networks
        self.q = DuelingQNet(n_actions=n_actions, dropout=dropout).to(self.device)
        self.q_target = DuelingQNet(n_actions=n_actions, dropout=dropout).to(self.device)
        self.q_target.load_state_dict(self.q.state_dict())

        # Optimizer with weight decay for regularization
        self.optim = optim.Adam(self.q.parameters(), lr=lr, weight_decay=1e-5)
        self.gamma = gamma

        # Exploration parameters
        self.eps = epsilon_start
        self.eps_end = epsilon_end
        self.eps_decay = epsilon_decay

        # Training parameters
        self.batch_size = batch_size
        self.target_update = target_update
        self.steps = 0
        self.n_actions = n_actions
        self.tau = tau
        self.use_soft_update = use_soft_update
        self.noise_std = noise_std

        # Experience replay components
        self.per_buffer = PERBuffer(capacity=100000, n_actions=n_actions,
                                    alpha=per_alpha, beta_start=per_beta_start,
                                    beta_frames=per_beta_frames)
        self.n_step = n_step
        self.nstep_buffer = NStepBuffer(n=n_step, gamma=gamma)

    def select_action(self, state: np.ndarray, legal_actions: List[int], eval_mode: bool = False) -> int:
        """
        Select action using epsilon-greedy policy with legal move masking

        Args:
            state: Current board observation
            legal_actions: List of legal action indices
            eval_mode: If True, disable exploration and noise

        Returns:
            Selected action index
        """
        current_eps = 0.0 if eval_mode else self.eps

        # Epsilon-greedy exploration
        if random.random() < current_eps:
            return random.choice(legal_actions)

        # Prepare state tensor
        st = torch.tensor(state, dtype=torch.float32, device=self.device)\
                .unsqueeze(0).permute(0, 3, 1, 2).contiguous()

        self.q.train(not eval_mode)

        with torch.no_grad():
            q = self.q(st).squeeze(0)

            # Add noise during training for exploration
            if not eval_mode and self.noise_std > 0:
                noise = torch.randn_like(q) * self.noise_std
                q = q + noise

        # Mask illegal actions
        mask = torch.full((self.n_actions,), -1e9, device=self.device)
        idx = torch.tensor(legal_actions, dtype=torch.long, device=self.device)
        mask[idx] = 0.0
        q_masked = q + mask

        return int(torch.argmax(q_masked).item())

    def _soft_update(self):
        """Soft update of target network parameters"""
        with torch.no_grad():
            for tp, p in zip(self.q_target.parameters(), self.q.parameters()):
                tp.data.lerp_((p.data), self.tau)

    def store_transition(self, state, action, reward, next_state, done, next_legal_mask):
        """Store experience in n-step buffer and replay buffer"""
        exp = NStepExp(state, action, reward, next_state, done, next_legal_mask)
        self.nstep_buffer.push(exp)

        # Pop n-step experience when buffer is full
        if self.nstep_buffer.can_pop():
            n_exp = self.nstep_buffer.pop()
            self.per_buffer.push(n_exp.state, n_exp.action, n_exp.reward,
                                 n_exp.next_state, n_exp.done, n_exp.mask)

        # Flush remaining experiences at episode end
        if done:
            for n_exp in self.nstep_buffer.flush_all():
                self.per_buffer.push(n_exp.state, n_exp.action, n_exp.reward,
                                     n_exp.next_state, n_exp.done, n_exp.mask)

    def update(self):
        """
        Perform one gradient update step

        Returns:
            Training loss value
        """
        if len(self.per_buffer) < self.batch_size:
            return 0.0

        self.q.train()
        self.q_target.eval()

        # Sample batch from replay buffer
        states, actions, rewards, next_states, dones, next_legal_masks, weights, idxs = \
            self.per_buffer.sample(self.batch_size)

        # Move to device and reshape
        states = states.to(self.device).permute(0, 3, 1, 2).contiguous()
        next_states = next_states.to(self.device).permute(0, 3, 1, 2).contiguous()
        actions = actions.to(self.device).long()
        rewards = rewards.to(self.device).float()
        dones = dones.to(self.device)
        next_legal_masks = next_legal_masks.to(self.device)
        weights = weights.to(self.device)
        idxs = idxs.to(self.device)

        # Current Q-values
        q_values = self.q(states)
        q_a = q_values.gather(1, actions.unsqueeze(1)).squeeze(1)

        # Double DQN target calculation
        with torch.no_grad():
            # Use online network to select actions
            next_q_online = self.q(next_states)
            next_q_online = next_q_online.masked_fill(~next_legal_masks, -1e9)
            next_actions = torch.argmax(next_q_online, dim=1, keepdim=True)

            # Use target network to evaluate actions
            next_q_target = self.q_target(next_states)
            next_q_target = next_q_target.masked_fill(~next_legal_masks, -1e9)
            next_q_target_a = next_q_target.gather(1, next_actions).squeeze(1)

            # Calculate target values
            target = rewards + (1 - dones.float()) * (self.gamma ** self.n_step) * next_q_target_a

        # Compute loss with importance sampling weights
        td_errors = target - q_a
        loss = (weights * F.smooth_l1_loss(q_a, target, reduction='none')).mean()

        # Backward pass with gradient clipping
        self.optim.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q.parameters(), 1.0)
        self.optim.step()

        # Update priorities in replay buffer
        new_prios = td_errors.abs() + 1e-6
        self.per_buffer.update_priorities(idxs, new_prios)

        # Update target network
        self.steps += 1
        if self.use_soft_update:
            self._soft_update()
        elif self.steps % self.target_update == 0:
            self.q_target.load_state_dict(self.q.state_dict())

        # Decay exploration
        self.eps = max(self.eps_end, self.eps * self.eps_decay)

        return float(loss.item())

    def save(self, path: str):
        """Save agent state to file"""
        os.makedirs(os.path.dirname(path), exist_ok=True)
        torch.save({
            "q": self.q.state_dict(),
            "q_target": self.q_target.state_dict(),
            "optim": self.optim.state_dict(),
            "eps": self.eps,
            "steps": self.steps
        }, path)

    def load(self, path: str):
        """Load agent state from file"""
        ckpt = torch.load(path, map_location=self.device)
        self.q.load_state_dict(ckpt["q"])
        self.q_target.load_state_dict(ckpt["q_target"])
        self.optim.load_state_dict(ckpt["optim"])
        self.eps = ckpt.get("eps", 0.1)
        self.steps = ckpt.get("steps", 0)

In [29]:
# SECTION 4: TRAINING SYSTEM

class BulletChessDDQNTrainer:
    """
    Complete training system for chess DDQN agent

    Features:
    - Self-play against configurable opponent
    - Progressive opponent strength curriculum
    - Comprehensive evaluation and logging
    - Model checkpointing and best model tracking
    - Detailed game statistics and analysis
    """

    def __init__(self, cfg: dict):
        self.cfg = cfg
        seed_everything(cfg.get("seed", 42))

        # Initialize environment
        self.env = BulletChessEnv(
            time_limit=cfg["env"]["time_limit"],
            increment=cfg["env"]["increment"],
            simulate_think=cfg["env"]["simulate_think"],
            think_lo=cfg["env"]["think_lo"],
            think_hi=cfg["env"]["think_hi"]
        )

        # Initialize agent
        self.agent = DDQNAgent(
            lr=cfg["agent"]["lr"],
            gamma=cfg["agent"]["gamma"],
            epsilon_start=cfg["agent"]["epsilon_start"],
            epsilon_end=cfg["agent"]["epsilon_end"],
            epsilon_decay=cfg["agent"]["epsilon_decay"],
            batch_size=cfg["agent"]["batch_size"],
            target_update=cfg["agent"]["target_update"],
            tau=cfg["agent"]["tau"],
            use_soft_update=cfg["agent"]["use_soft_update"],
            n_actions=4096,
            per_alpha=cfg["replay"]["per_alpha"],
            per_beta_start=cfg["replay"]["per_beta_start"],
            per_beta_frames=cfg["replay"]["per_beta_frames"],
            n_step=cfg["agent"]["n_step"],
            dropout=cfg["agent"]["dropout"],
            noise_std=cfg["agent"]["noise_std"]
        )

        self.setup_logging()

        # Training statistics
        self.best_win_rate = -1.0
        self.stats = {
            "wins": 0, "losses": 0, "draws": 0,
            "wins_white": 0, "wins_black": 0,
            "losses_white": 0, "losses_black": 0,
            "illegal": 0
        }
        self.ended_by = Counter()

        # Opponent curriculum parameters
        self.opp_strength_start = cfg["opponent"]["strength_start"]
        self.opp_strength_end = cfg["opponent"]["strength_end"]

    def setup_logging(self):
        """Initialize logging system with file and console output"""
        os.makedirs(self.cfg["paths"]["models"], exist_ok=True)
        os.makedirs(self.cfg["paths"]["logs"], exist_ok=True)

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        log_file = os.path.join(self.cfg["paths"]["logs"], f"ddqn_truly_fixed_{timestamp}.log")

        # Clear existing handlers
        for h in logging.root.handlers[:]:
            logging.root.removeHandler(h)

        logging.basicConfig(
            level=logging.INFO,
            format="%(asctime)s - %(levelname)s - %(message)s",
            handlers=[logging.FileHandler(log_file), logging.StreamHandler()]
        )
        self.logger = logging.getLogger("ddqn")

    def select_opponent_move(self, legal_actions: List[int], strength: float) -> int:
        """
        Rule-based opponent with configurable strength

        Args:
            legal_actions: List of legal action indices
            strength: Opponent skill level (0.0 = random, 1.0 = perfect tactical play)

        Returns:
            Selected action index
        """
        # Random move with probability (1 - strength)
        if random.random() > strength:
            return random.choice(legal_actions)

        board = self.env.state.board
        best = None
        best_score = -1e9

        # Define important squares for positional evaluation
        center = {chess.D4, chess.E4, chess.D5, chess.E5}
        ext_center = {
            chess.C3, chess.C4, chess.C5, chess.C6,
            chess.D3, chess.D6, chess.E3, chess.E6,
            chess.F3, chess.F4, chess.F5, chess.F6
        }

        # Evaluate each legal move
        for a in legal_actions:
            from_sq, to_sq = a // 64, a % 64
            move = chess.Move(from_sq, to_sq)
            score = 0.0

            # Prioritize captures by material value
            if board.is_capture(move):
                captured = board.piece_at(to_sq)
                if captured:
                    pv = {chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3,
                          chess.ROOK: 5, chess.QUEEN: 9, chess.KING: 0}
                    score += pv.get(captured.piece_type, 0) * 10

            # Test move consequences
            tmp = board.copy()
            tmp.push(move)

            # Bonus for checks and checkmate
            if tmp.is_check():
                score += 3.0
            if tmp.is_checkmate():
                score += 1000.0

            # Positional bonuses for center control
            if to_sq in center:
                score += 1.0
            elif to_sq in ext_center:
                score += 0.5

            # Avoid hanging pieces (basic safety check)
            piece = board.piece_at(from_sq)
            if piece and not board.is_capture(move):
                tmp2 = board.copy()
                tmp2.push(move)
                if tmp2.is_attacked_by(not tmp2.turn, to_sq):
                    pv = {chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3,
                          chess.ROOK: 5, chess.QUEEN: 9, chess.KING: 0}
                    score -= pv.get(piece.piece_type, 0) * 3

            # Add small random factor for variety
            score += random.uniform(-0.1, 0.1)

            if score > best_score:
                best_score = score
                best = a

        return best if best is not None else random.choice(legal_actions)

    def play_one(self, is_training: bool, episode_idx: int, total_episodes: int,
                 force_eval_strength: float = None) -> Dict:
        """
        Play one complete game between agent and opponent

        Args:
            is_training: Whether to collect training data
            episode_idx: Current episode number for curriculum
            total_episodes: Total episodes for curriculum scheduling
            force_eval_strength: Override opponent strength for evaluation

        Returns:
            Game result dictionary with statistics
        """
        state = self.env.reset()
        total_reward = 0.0
        steps = 0
        ended_by = "unknown"

        # Randomly assign colors
        agent_is_white = random.choice([True, False])

        # Calculate opponent strength from curriculum
        if force_eval_strength is not None:
            opp_strength = force_eval_strength
        else:
            opp_strength = linear_anneal(self.opp_strength_start, self.opp_strength_end,
                                         episode_idx, total_episodes)

        # Handle first move if opponent plays first
        if (self.env.state.board.turn and not agent_is_white) or \
           ((not self.env.state.board.turn) and agent_is_white):
            legal_opp = self.env.get_legal_actions()
            if legal_opp:
                opp_action = self.select_opponent_move(legal_opp, opp_strength)
                state, _, done, info = self.env.step(opp_action)
                steps += 1
                if done:
                    ended_by = self._map_reason(info.get("reason", "unknown"))
                    return self._finalize_episode(total_reward, steps, agent_is_white, ended_by)

        # Reset n-step buffer for new episode
        if is_training:
            self.agent.nstep_buffer.reset()

        # Main game loop
        while steps < self.cfg["training"]["max_moves"] and not self.env.is_game_over():
            # Agent's turn
            legal_agent = self.env.get_legal_actions()
            if not legal_agent:
                ended_by = "no_legal_agent"
                break

            eval_mode = not is_training
            action = self.agent.select_action(state, legal_agent, eval_mode=eval_mode)
            next_state_after_agent, r_agent_raw, done_agent, info_agent = self.env.step(action)
            steps += 1

            # Adjust reward sign based on agent's color
            r_agent = r_agent_raw if agent_is_white else -r_agent_raw

            if done_agent:
                ended_by = self._map_reason(info_agent.get("reason", "unknown"))
                mask = np.zeros((self.agent.n_actions,), dtype=np.bool_)
                if is_training:
                    self.agent.store_transition(state, action, r_agent, next_state_after_agent, True, mask)
                total_reward += r_agent
                state = next_state_after_agent
                break

            # Opponent's turn
            legal_opp = self.env.get_legal_actions()
            if not legal_opp:
                ended_by = "no_legal_opponent"
                mask = np.zeros((self.agent.n_actions,), dtype=np.bool_)
                if is_training:
                    self.agent.store_transition(state, action, r_agent, next_state_after_agent, True, mask)
                total_reward += r_agent
                state = next_state_after_agent
                break

            opp_action = self.select_opponent_move(legal_opp, opp_strength)
            next_state, r_opp_raw, done_opp, info_opp = self.env.step(opp_action)
            steps += 1

            if done_opp:
                # Calculate terminal bonus based on game result
                result = self.env.get_result()
                terminal_bonus = 0.0
                if result == "1-0":
                    terminal_bonus = 15.0 if agent_is_white else -15.0
                elif result == "0-1":
                    terminal_bonus = 15.0 if not agent_is_white else -15.0
                else:
                    terminal_bonus = 0.0

                final_r = r_agent + terminal_bonus
                ended_by = self._map_reason(info_opp.get("reason", "unknown"))

                mask = np.zeros((self.agent.n_actions,), dtype=np.bool_)
                if is_training:
                    self.agent.store_transition(state, action, final_r, next_state, True, mask)
                total_reward += final_r
                state = next_state
                break
            else:
                # Continue game - prepare legal action mask for next state
                legal_next_agent = self.env.get_legal_actions()
                mask = np.zeros((self.agent.n_actions,), dtype=np.bool_)
                if legal_next_agent:
                    mask[np.array(legal_next_agent, dtype=np.int32)] = True

                if is_training:
                    self.agent.store_transition(state, action, r_agent, next_state, False, mask)

                total_reward += r_agent
                state = next_state

        # Handle move limit penalty
        if steps >= self.cfg["training"]["max_moves"] and ended_by == "unknown":
            ended_by = "move_limit"
            if is_training:
                final_pen = self.cfg["training"]["move_limit_penalty"]
                mask = np.zeros((self.agent.n_actions,), dtype=np.bool_)
                self.agent.store_transition(state, action if 'action' in locals() else 0,
                                          final_pen, state, True, mask)
                total_reward += final_pen

        return self._finalize_episode(total_reward, steps, agent_is_white, ended_by)

    def _map_reason(self, reason: str) -> str:
        """Map detailed termination reasons to categories"""
        if "checkmate" in reason:
            return "mate"
        if "time_flag" in reason:
            return "time_flag"
        if "draw" in reason:
            return "draw"
        if "illegal" in reason:
            return "illegal"
        if "timeout_or_resignation" in reason:
            return "timeout_or_resignation"
        return reason

    def _finalize_episode(self, total_reward, steps, agent_is_white, ended_by) -> Dict:
        """Process episode end and update statistics"""
        result = self.env.get_result()
        outcome = "draw"

        # Determine outcome from agent's perspective
        if result == "1-0":
            outcome = "win" if agent_is_white else "loss"
        elif result == "0-1":
            outcome = "win" if not agent_is_white else "loss"

        # Update statistics
        if outcome == "win":
            self.stats["wins"] += 1
            if agent_is_white:
                self.stats["wins_white"] += 1
            else:
                self.stats["wins_black"] += 1
        elif outcome == "loss":
            self.stats["losses"] += 1
            if agent_is_white:
                self.stats["losses_white"] += 1
            else:
                self.stats["losses_black"] += 1
        else:
            self.stats["draws"] += 1

        if ended_by:
            self.ended_by[ended_by] += 1

        return {
            "reward": total_reward,
            "steps": steps,
            "outcome": outcome,
            "result": result,
            "agent_white": agent_is_white,
            "ended_by": ended_by
        }

    def train(self):
        """
        Main training loop with curriculum learning and evaluation

        Training features:
        - Progressive opponent strength curriculum
        - Regular evaluation against stronger opponents
        - Best model tracking and checkpointing
        - Comprehensive logging of training progress
        """
        episodes = self.cfg["training"]["episodes"]
        warmup = self.cfg["training"]["warmup"]
        log_freq = self.cfg["training"]["log_freq"]
        eval_freq = self.cfg["training"]["eval_freq"]
        save_freq = self.cfg["training"]["save_freq"]

        losses = deque(maxlen=200)

        for ep in range(1, episodes + 1):
            # Play training game
            res = self.play_one(is_training=True, episode_idx=ep, total_episodes=episodes)

            # Perform learning update after warmup period
            loss_val = 0.0
            if ep >= warmup:
                loss_val = self.agent.update()
                if loss_val:
                    losses.append(loss_val)

            # Calculate statistics
            total_games = self.stats["wins"] + self.stats["losses"] + self.stats["draws"]
            wr_all = self.stats["wins"] / max(1, total_games)  # Win rate including draws
            wr_true = self.stats["wins"] / max(1, (self.stats["wins"] + self.stats["losses"]))  # Win rate excluding draws

            # Periodic logging
            if ep % log_freq == 0 or ep == 1:
                opp_strength = linear_anneal(self.opp_strength_start, self.opp_strength_end,
                                           ep, episodes)
                ml_rate = self.ended_by.get("move_limit", 0) / max(1, total_games)
                self.logger.info(
                    f"Ep {ep:4d}/{episodes} | {res['outcome']:4s} | "
                    f"R:{res['reward']:6.2f} | M:{res['steps']:3d} | "
                    f"WR_all:{wr_all:.3f} | WR_true:{wr_true:.3f} | "
                    f"L:{(np.mean(losses) if losses else 0):.4f} | "
                    f"eps:{self.agent.eps:.3f} | buf:{len(self.agent.per_buffer)} | "
                    f"Ww:{self.stats['wins_white']} Wb:{self.stats['wins_black']} "
                    f"Lw:{self.stats['losses_white']} Lb:{self.stats['losses_black']} | "
                    f"opp_str:{opp_strength:.3f} | move_limit_rate:{ml_rate:.3f}"
                )

            # Periodic evaluation
            if ep % eval_freq == 0:
                eval_strength = min(0.25, opp_strength + 0.05)
                wr, wr_true_eval = self.evaluate(self.cfg["training"]["eval_games"],
                                                ep, episodes, eval_strength)
                # Save best model
                if wr > self.best_win_rate:
                    self.best_win_rate = wr
                    best_path = os.path.join(self.cfg["paths"]["models"], f"best_ddqn_truly_fixed_ep{ep}.pth")
                    self.agent.save(best_path)
                    self.logger.info(f"New best model saved (win_rate={wr:.3f}) -> {best_path}")

            # Periodic checkpointing
            if ep % save_freq == 0:
                ckpt_path = os.path.join(self.cfg["paths"]["models"], f"ckpt_ddqn_truly_fixed_ep{ep}.pth")
                self.agent.save(ckpt_path)
                self.logger.info(f"Checkpoint saved -> {ckpt_path}")

        # Final comprehensive evaluation across multiple opponent strengths
        self.logger.info("=== FINAL EVALUATION ===")
        for strength in [0.1, 0.15, 0.2, 0.25, 0.3]:
            wr, wr_true = self.evaluate(30, episodes, episodes, force_strength=strength)
            self.logger.info(f"vs {strength:.2f} strength: WR={wr:.3f} (true={wr_true:.3f})")

    def evaluate(self, n_games: int, ep: int, total_episodes: int,
                 force_strength: float = None) -> Tuple[float, float]:
        """
        Evaluate agent performance against opponent

        Args:
            n_games: Number of evaluation games
            ep: Current episode for curriculum
            total_episodes: Total episodes for curriculum
            force_strength: Override opponent strength

        Returns:
            (win_rate_all, win_rate_true) - with and without draws
        """
        wins = 0
        draws = 0
        losses = 0
        lens = []
        wins_white = wins_black = 0
        losses_white = losses_black = 0

        # Disable exploration during evaluation
        old_eps = self.agent.eps
        self.agent.eps = 0.0

        eval_strength = force_strength or min(0.25,
            linear_anneal(self.opp_strength_start, self.opp_strength_end, ep, total_episodes) + 0.05)

        # Play evaluation games
        for i in range(n_games):
            res = self.play_one(is_training=False, episode_idx=ep,
                              total_episodes=total_episodes,
                              force_eval_strength=eval_strength)

            lens.append(res["steps"])

            # Log first few games for debugging
            if i < 3:
                self.logger.info(f"EVAL Game {i+1}: {res['outcome']} in {res['steps']} moves, "
                                f"ended_by: {res['ended_by']}, agent_white: {res['agent_white']}")

            # Update evaluation statistics
            if res["outcome"] == "win":
                wins += 1
                if res["agent_white"]:
                    wins_white += 1
                else:
                    wins_black += 1
            elif res["outcome"] == "loss":
                losses += 1
                if res["agent_white"]:
                    losses_white += 1
                else:
                    losses_black += 1
            else:
                draws += 1

        # Restore exploration
        self.agent.eps = old_eps

        # Calculate win rates
        wr_all = wins / max(1, (wins + losses + draws))
        wr_true = wins / max(1, (wins + losses))

        self.logger.info(f"[EVAL] vs {eval_strength:.2f} | WR_all {wr_all:.3f} | WR_true {wr_true:.3f} | "
                         f"W:{wins} L:{losses} D:{draws} | "
                         f"W_white:{wins_white} W_black:{wins_black} | "
                         f"L_white:{losses_white} L_black:{losses_black} | "
                         f"len:{np.mean(lens):.1f}")
        return wr_all, wr_true


def get_truly_fixed_cfg():
    """
    Configuration dictionary for training hyperparameters

    Contains all hyperparameters for:
    - Environment setup (time controls, thinking simulation)
    - Agent architecture and learning parameters
    - Experience replay configuration
    - Training schedule and evaluation
    - Opponent curriculum
    - File paths for models and logs
    """
    return {
        "seed": 42,
        "env": {
            "time_limit": 60,      # Bullet chess time limit in seconds
            "increment": 0.0,      # Time increment per move
            "simulate_think": True, # Simulate realistic thinking time
            "think_lo": 0.80,      # Minimum thinking time
            "think_hi": 1.60       # Maximum thinking time
        },
        "agent": {
            "lr": 1e-4,            # Learning rate
            "gamma": 0.99,         # Discount factor
            "epsilon_start": 1.0,  # Initial exploration rate
            "epsilon_end": 0.05,   # Final exploration rate
            "epsilon_decay": 0.9996, # Exploration decay rate
            "batch_size": 64,      # Mini-batch size
            "target_update": 2000, # Hard target update frequency
            "tau": 0.002,          # Soft target update rate
            "use_soft_update": True, # Use soft vs hard target updates
            "n_step": 3,           # N-step learning horizon
            "dropout": 0.1,        # Network dropout rate
            "noise_std": 0.01      # Action noise for exploration
        },
        "replay": {
            "capacity": 100000,    # Replay buffer size
            "per_alpha": 0.6,      # PER priority exponent
            "per_beta_start": 0.4, # PER importance sampling start
            "per_beta_frames": 1_000_000 # PER beta annealing frames
        },
        "training": {
            "episodes": 1500,      # Total training episodes
            "max_moves": 120,      # Maximum moves per game
            "move_limit_penalty": -3.0, # Penalty for reaching move limit
            "warmup": 100,         # Episodes before learning starts
            "log_freq": 50,        # Logging frequency
            "eval_freq": 150,      # Evaluation frequency
            "eval_games": 30,      # Games per evaluation
            "save_freq": 300       # Model saving frequency
        },
        "opponent": {
            "strength_start": 0.0, # Initial opponent strength
            "strength_end": 0.3    # Final opponent strength
        },
        "paths": {
            "models": "models_ddqn_truly_fixed", # Model save directory
            "logs": "logs_ddqn_truly_fixed"      # Log save directory
        }
    }


def main():
    """
    Main training entry point

    Performs:
    1. Configuration setup
    2. Environment validation
    3. Training execution
    """
    cfg = get_truly_fixed_cfg()
    trainer = BulletChessDDQNTrainer(cfg)

    # Quick environment test to verify setup
    print("=== Environment Test ===")
    env = BulletChessEnv()
    state = env.reset()
    print(f"Initial state shape: {state.shape}")
    print(f"Initial board: {env.state.board}")
    print(f"Legal moves: {len(env.get_legal_actions())}")
    print(f"Game over: {env.is_game_over()}")
    print("=== Starting Training ===")

    trainer.train()


if __name__ == "__main__":
    main()

2025-07-24 20:18:09,494 - INFO - Ep    1/1500 | win  | R: 15.13 | M:102 | WR_all:1.000 | WR_true:1.000 | L:0.0000 | eps:1.000 | buf:51 | Ww:1 Wb:0 Lw:0 Lb:0 | opp_str:0.000 | move_limit_rate:0.000


=== Environment Test ===
Initial state shape: (8, 8, 15)
Initial board: r n b q k b n r
p p p p p p p p
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
P P P P P P P P
R N B Q K B N R
Legal moves: 20
Game over: False
=== Starting Training ===


2025-07-24 20:18:12,974 - INFO - Ep   50/1500 | win  | R: 16.45 | M:100 | WR_all:0.460 | WR_true:0.460 | L:0.0000 | eps:1.000 | buf:2409 | Ww:8 Wb:15 Lw:15 Lb:12 | opp_str:0.010 | move_limit_rate:0.000
2025-07-24 20:18:17,928 - INFO - Ep  100/1500 | loss | R:-14.63 | M: 88 | WR_all:0.470 | WR_true:0.470 | L:1.1541 | eps:1.000 | buf:4890 | Ww:14 Wb:33 Lw:31 Lb:22 | opp_str:0.020 | move_limit_rate:0.000
2025-07-24 20:19:05,127 - INFO - Ep  150/1500 | loss | R:-19.95 | M:101 | WR_all:0.460 | WR_true:0.460 | L:0.7824 | eps:0.980 | buf:7301 | Ww:25 Wb:44 Lw:46 Lb:35 | opp_str:0.030 | move_limit_rate:0.000
2025-07-24 20:19:05,652 - INFO - EVAL Game 1: loss in 97 moves, ended_by: time_flag, agent_white: True
2025-07-24 20:19:06,218 - INFO - EVAL Game 2: win in 96 moves, ended_by: time_flag, agent_white: True
2025-07-24 20:19:06,828 - INFO - EVAL Game 3: loss in 98 moves, ended_by: time_flag, agent_white: False
2025-07-24 20:19:21,962 - INFO - [EVAL] vs 0.08 | WR_all 0.433 | WR_true 0.433 | W: